In [1]:
# Cell 1 — Imports and parameters
from pathlib import Path
from typing import List, Optional, Tuple
import numpy as np
from skimage import io
from skimage.measure import regionprops, label
from skimage.transform import resize
import pandas as pd
from tqdm import tqdm
import shutil

project_root = Path("/Users/ashi/github/cm4ai_codefest2025")

# Masks from Notebook 1
masks_root  = project_root / "analysis" / "cellpose_results2"

# Raw images
img_root    = project_root / "data"

# Output crops and CSV
output_root = project_root / "analysis" / "cell_crops"

# Channels + single reference channel
channels: List[str] = ["red", "yellow", "blue", "green"]
ref_channel = "yellow"

# Crop settings
crop_size: Optional[int] = 640   # None -> tight bbox; else fixed squares
bbox_pad_px: int = 0             # only used when crop_size is None
image_exts = [".tif", ".tiff", ".png", ".jpg", ".jpeg"]

# CSV schema
csv_name = "pred_cell.csv"
csv_columns = ["r_image","y_image","b_image","g_image","output_folder","output_prefix","cell_id","image_id"]


In [2]:
# Cell 2 — Helpers
def ensure_fresh_output(root: Path, chans: List[str]):
    if root.exists():
        shutil.rmtree(root)
    for ch in chans:
        (root / ch).mkdir(parents=True, exist_ok=True)

def read_mask(p: Path) -> np.ndarray:
    m = io.imread(p)
    if m.ndim > 2:
        m = m[..., 0]
    return m

def labeled_from_mask(mask: np.ndarray) -> np.ndarray:
    u = np.unique(mask)
    return mask.astype(np.int32) if (u.size > 2 and u.max() > 1) else label(mask > 0)

def square_window(center_r: float, center_c: float, size: int, img_shape: Tuple[int, int]) -> Tuple[int, int, int, int]:
    H, W = img_shape
    half = size // 2
    r0 = int(round(center_r)) - half
    c0 = int(round(center_c)) - half
    r0 = max(r0, 0); c0 = max(c0, 0)
    r1 = min(r0 + size, H)
    c1 = min(c0 + size, W)
    if r1 - r0 < size: r0 = max(r1 - size, 0)
    if c1 - c0 < size: c0 = max(c1 - size, 0)
    return (r0, c0, r1, c1)

def safe_crop(img: np.ndarray, win: Tuple[int, int, int, int], pad_value: int | float = 0) -> np.ndarray:
    r0, c0, r1, c1 = win
    H, W = img.shape[:2]
    sr0, sc0 = max(r0, 0), max(c0, 0)
    sr1, sc1 = min(r1, H), min(c1, W)
    out_h, out_w = r1 - r0, c1 - c0
    out_shape = (out_h, out_w) if img.ndim == 2 else (out_h, out_w, img.shape[2])
    out = np.full(out_shape, pad_value, dtype=img.dtype)
    dr, dc = sr0 - r0, sc0 - c0
    out[dr:dr + (sr1 - sr0), dc:dc + (sc1 - sc0)] = img[sr0:sr1, sc0:sc1]
    return out

def read_image_any(basepath: Path):
    for ext in image_exts:
        p = basepath.with_suffix(ext)
        if p.exists():
            return io.imread(p), p
    return None, None


In [3]:
# Cell 3 — Collect yellow masks
ref_dir = masks_root / ref_channel / "png"
mask_files = sorted(ref_dir.glob("*_masks.png"))
if not mask_files:
    raise FileNotFoundError(f"No *_masks.png in {ref_dir}. Run Notebook 1 for {ref_channel} first.")
print(f"Found {len(mask_files)} yellow reference masks in {ref_dir}")


Found 10 yellow reference masks in /Users/ashi/github/cm4ai_codefest2025/analysis/cellpose_results2/yellow/png


In [4]:
# Cell 4 — Synchronized cropping from yellow ROIs across all channels
ensure_fresh_output(output_root, channels)
rows = []

for mpath in tqdm(mask_files, desc="Cropping from yellow masks"):
    base = mpath.stem.replace("_masks", "")
    mask = read_mask(mpath)
    labeled = labeled_from_mask(mask)
    regions = regionprops(labeled)
    if not regions:
        continue

    # load the matching raw image for every channel by basename
    imgs = {}
    for ch in channels:
        img, _ = read_image_any(img_root / ch / base)
        imgs[ch] = img

    for i, reg in enumerate(regions, 1):
        if crop_size is not None:
            win = square_window(reg.centroid[0], reg.centroid[1], crop_size, mask.shape)
        else:
            r0, c0, r1, c1 = reg.bbox
            win = (r0 - bbox_pad_px, c0 - bbox_pad_px, r1 + bbox_pad_px, c1 + bbox_pad_px)

        out_paths = {}
        for ch in channels:
            if imgs[ch] is None:
                continue
            crop = safe_crop(imgs[ch], win)
            if crop_size is not None and (crop.shape[0] != crop_size or crop.shape[1] != crop_size):
                crop = resize(crop, (crop_size, crop_size), order=1, mode="edge", preserve_range=True, anti_aliasing=True).astype(imgs[ch].dtype)
            out_p = output_root / ch / f"{base}_cell{i}.png"
            io.imsave(out_p, crop, check_contrast=False)
            out_paths[ch] = str(out_p)

        rows.append([
            out_paths.get("red",""),
            out_paths.get("yellow",""),
            out_paths.get("blue",""),
            out_paths.get("green",""),
            str(output_root),
            f"{base}_",
            str(i),
            base
        ])

df = pd.DataFrame(rows, columns=csv_columns)
csv_path = output_root / csv_name
df.to_csv(csv_path, index=False)
print(f"Wrote {len(df)} rows to {csv_path}")


Cropping from yellow masks: 100%|██████████| 10/10 [00:32<00:00,  3.25s/it]

Wrote 2364 rows to /Users/ashi/github/cm4ai_codefest2025/analysis/cell_crops/pred_cell.csv


In [5]:
# Cell 5 — Post run checks
counts = {ch: len(list((output_root / ch).glob("*.png"))) for ch in channels if (output_root / ch).exists()}
print("Crop counts per channel:", counts)
if len(counts) == len(channels) and len(set(counts.values())) == 1:
    print("Counts match across channels. Synchronized cropping succeeded.")
else:
    print("Counts differ. Likely missing raw images or mismatched basenames.")


Crop counts per channel: {'red': 0, 'yellow': 2364, 'blue': 0, 'green': 0}
Counts differ. Likely missing raw images or mismatched basenames.
